# SegFormer-B2 vs U-Net (ResNet-50) on LoveDA

Single-notebook version of the `dl_research` package (generated by `scripts/build_notebook.py`; the package is the source of truth).

**Research questions**
1. Under one shared training/evaluation protocol, how do the models compare in mIoU?
2. How does the gap change with 100% / 25% / 10% of the training labels?
3. Does relative performance differ between the urban and rural domains?
4. How do parameters, compute and speed relate to segmentation quality?

> **No results are claimed here.** Hypotheses (e.g. that the transformer encoder may help more with few labels) are
> plans to be tested, not findings. Numbers appear only after you run the study cells below.

**Protocol (identical for both models):** 512×512 random crops, flips + k·90° rotations, cross-entropy (ignore 255), AdamW, matched
epochs (scaled by 1/fraction so each fraction gets ~equal optimizer steps), ImageNet-pretrained encoders, seeds 0–2. Label
fractions are seeded, domain-stratified, nested subsets of whole images. Learning rates are tuned on the validation split only.

**Limits:** LoveDA test labels are not public, so everything is reported on the **validation** split, which is also used for LR and
checkpoint selection (optimistically biased). Pretrained-weight downloads happen on first non-smoke run. Run order: setup →
smoke test → dataset → tuning → study → aggregate. The expensive stages are behind `RUN_*` flags (default off).

## 1. Setup

In [ ]:
# %pip install torch numpy pyyaml pillow pandas matplotlib segmentation-models-pytorch transformers
import argparse, json, math, platform, random, shutil, subprocess, sys, time, zipfile  # noqa: E401
import copy
import numpy as np, torch  # noqa: E401
print(sys.version.split()[0], "torch", torch.__version__)

## 2. Library code
One cell per module of the `dl_research` package, in dependency order. Run all of them once.

### Configuration  
Dataclass holding every setting; YAML/override handling is replaced by `build_config` below.

In [ ]:
"""Experiment configuration: YAML files are merged left to right, then `key=value` overrides."""

import copy
from dataclasses import asdict, dataclass, field, fields
from pathlib import Path
from typing import Any

import yaml

MODELS = ("unet_r50", "segformer_b2")


@dataclass
class Config:
    model: str = "unet_r50"
    dataset: str = "loveda"
    data_root: str = "data/LoveDA"
    out_dir: str = "experiments"
    seed: int = 0
    fraction: float = 1.0
    crop_size: int = 512
    crops_per_image: int = 1
    batch_size: int = 8
    eval_batch_size: int = 2
    num_workers: int = 4
    epochs: int = 30
    scale_epochs_by_fraction: bool = True
    num_evals: int = 10
    optimizer: str = "adamw"
    lr: float | None = None
    tuned_lr_file: str = "configs/tuned_lr.json"
    weight_decay: float = 0.01
    warmup_frac: float = 0.05
    amp: bool = True
    eval_amp: bool = False            # fp16 autocast during validation (CUDA only); same for both models
    prune_last_checkpoint: bool = False  # delete checkpoints/last.pt once a run is complete (saves disk)
    pretrained: bool = True
    encoder: str = "resnet50"
    segformer_checkpoint: str = "nvidia/mit-b2"
    ignore_index: int = 255
    predictions_max_images: int = 20
    tune_fraction: float = 1.0
    tune_epochs: int = 10
    tune_seed: int = 0
    lr_grids: dict = field(default_factory=dict)
    bench_warmup: int = 10
    bench_iters: int = 50
    bench_batch_size: int = 1
    smoke: bool = False
    device: str = "auto"

    def validate(self) -> "Config":
        if self.model not in MODELS:
            raise ValueError(f"model must be one of {MODELS}, got {self.model!r}")
        if not 0 < self.fraction <= 1:
            raise ValueError(f"fraction must be in (0, 1], got {self.fraction}")
        if self.crop_size % 32:
            raise ValueError(f"crop_size must be a multiple of 32, got {self.crop_size}")
        if self.optimizer != "adamw":
            raise ValueError("only optimizer: adamw is implemented")
        for name in ("epochs", "batch_size", "eval_batch_size", "crops_per_image", "num_evals"):
            if getattr(self, name) < 1:
                raise ValueError(f"{name} must be >= 1")
        return self

    @property
    def total_epochs(self) -> int:
        """Epochs actually run. With scaling, every fraction gets (about) the same number of steps."""
        if self.scale_epochs_by_fraction:
            return max(1, round(self.epochs / self.fraction))
        return self.epochs

    @property
    def run_name(self) -> str:
        return f"{self.model}_f{self.fraction:g}_s{self.seed}"

    @property
    def run_dir(self) -> Path:
        return Path(self.out_dir) / self.run_name


def _deep_merge(base: dict, new: dict) -> dict:
    out = copy.deepcopy(base)
    for k, v in new.items():
        out[k] = _deep_merge(out[k], v) if isinstance(v, dict) and isinstance(out.get(k), dict) else v
    return out


def build_config(raw: dict[str, Any]) -> Config:
    known = {f.name for f in fields(Config)}
    unknown = set(raw) - known
    if unknown:
        raise ValueError(f"unknown config keys: {sorted(unknown)}")
    # YAML 1.1 reads `1e-3` as a string; coerce float-typed fields explicitly.
    for name in ("lr", "fraction", "weight_decay", "warmup_frac", "tune_fraction"):
        if isinstance(raw.get(name), (str, int)) and not isinstance(raw.get(name), bool):
            raw[name] = float(raw[name])
    return Config(**raw).validate()


def load_config(paths: list[str] | None = None, overrides: list[str] | None = None) -> Config:
    raw: dict[str, Any] = {}
    for p in paths or []:
        raw = _deep_merge(raw, yaml.safe_load(Path(p).read_text()) or {})
    for item in overrides or []:
        key, sep, value = item.partition("=")
        if not sep:
            raise ValueError(f"override must look like key=value, got {item!r}")
        raw[key.strip()] = yaml.safe_load(value)
    return build_config(raw)


def dump_config(cfg: Config, path: Path) -> None:
    path.write_text(yaml.safe_dump(asdict(cfg), sort_keys=True))


def read_config(path: Path) -> Config:
    return build_config(yaml.safe_load(Path(path).read_text()))


### Utilities  
Seeding, device selection, environment info, JSON helpers.

In [ ]:
import json
import platform
import random
import subprocess
import sys
from importlib import metadata
from pathlib import Path

import numpy as np
import torch


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Best effort: cuDNN determinism; some ops (e.g. bilinear upsampling backward) stay nondeterministic on GPU.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def pick_device(name: str = "auto") -> torch.device:
    if name != "auto":
        return torch.device(name)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def sync(device: torch.device) -> None:
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps":
        torch.mps.synchronize()


def _pkg(name: str) -> str | None:
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None


def _run(cmd: list[str]) -> str | None:
    try:
        return subprocess.run(cmd, capture_output=True, text=True, timeout=5).stdout.strip() or None
    except Exception:
        return None


def env_info(device: torch.device) -> dict:
    if device.type == "cuda":
        dev_name = torch.cuda.get_device_name(device)
    elif device.type == "mps":
        dev_name = "Apple MPS (" + (_run(["sysctl", "-n", "machdep.cpu.brand_string"]) or platform.machine()) + ")"
    else:
        dev_name = _run(["sysctl", "-n", "machdep.cpu.brand_string"]) or platform.processor() or platform.machine()
    return {
        "device": device.type,
        "device_name": dev_name,
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "cudnn": torch.backends.cudnn.version(),
        "segmentation_models_pytorch": _pkg("segmentation-models-pytorch"),
        "transformers": _pkg("transformers"),
        "torchvision": _pkg("torchvision"),
        "numpy": np.__version__,
        "torch_threads": torch.get_num_threads(),
        "git_commit": _run(["git", "rev-parse", "HEAD"]),
    }


def write_json(path: Path, obj) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2))
    tmp.replace(path)


def read_json(path: Path):
    return json.loads(Path(path).read_text())


### Data core  
Dataset registry, deterministic domain-stratified label-fraction subsets, augmentation, `SegDataset`, `EpochSampler`.

In [ ]:
"""Dataset-agnostic pieces: sample records, the dataset registry, deterministic subsets, SegDataset.

A new dataset (e.g. DeepGlobe) only needs a `DatasetSpec` registered in `DATASETS`; nothing here or in
train/evaluate changes.
"""

from dataclasses import dataclass
from pathlib import Path
from typing import Callable

import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset, Sampler

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)
IGNORE_INDEX = 255


@dataclass(frozen=True)
class Sample:
    image: str
    mask: str | None  # None when labels are unavailable (LoveDA test)
    domain: str
    id: str


@dataclass(frozen=True)
class DatasetSpec:
    name: str
    class_names: tuple[str, ...]
    domains: tuple[str, ...]
    decode_lut: np.ndarray  # raw mask value (0..255) -> train id (0..C-1) or IGNORE_INDEX
    encode_lut: np.ndarray  # train id -> raw mask value, for writing predictions
    scan: Callable[[Path, str, bool], list[Sample]]  # (root, split, require_labels) -> samples
    make_synthetic: Callable[..., None] | None = None

    @property
    def num_classes(self) -> int:
        return len(self.class_names)


DATASETS: dict[str, DatasetSpec] = {}


def register(spec: DatasetSpec) -> DatasetSpec:
    DATASETS[spec.name] = spec
    return spec


def get_spec(name: str) -> DatasetSpec:

    if name not in DATASETS:
        raise KeyError(f"unknown dataset {name!r}; registered: {sorted(DATASETS)}")
    return DATASETS[name]


def select_fraction(samples: list[Sample], fraction: float, seed: int) -> list[Sample]:
    """Deterministic label-fraction subset of training IMAGES, stratified by domain.

    Each domain is shuffled with a seeded permutation and its first `round(fraction * n)` images are kept, so
    (a) every domain keeps its proportion, (b) the result depends only on (samples, fraction, seed), and
    (c) subsets are nested: the 10% subset is contained in the 25% subset, which is contained in the 100% set.
    Class balance is NOT controlled.
    """
    if not 0 < fraction <= 1:
        raise ValueError(f"fraction must be in (0, 1], got {fraction}")
    out: list[Sample] = []
    for di, dom in enumerate(sorted({s.domain for s in samples})):
        group = sorted((s for s in samples if s.domain == dom), key=lambda s: s.id)
        n = len(group) if fraction == 1 else max(1, round(fraction * len(group)))
        perm = np.random.default_rng([seed, di]).permutation(len(group))
        out += [group[i] for i in sorted(perm[:n])]
    return out


def decode_mask(raw: np.ndarray, spec: DatasetSpec) -> np.ndarray:
    """Raw mask -> train ids; invalid/no-data raw values become IGNORE_INDEX."""
    if raw.ndim == 3:
        raw = raw[..., 0]
    return spec.decode_lut[raw]


def _augment(img: np.ndarray, mask: np.ndarray, crop: int, rng: np.random.Generator):
    h, w = mask.shape
    if h < crop or w < crop:  # pad: image with 0, mask with ignore
        ph, pw = max(0, crop - h), max(0, crop - w)
        img = np.pad(img, ((0, ph), (0, pw), (0, 0)))
        mask = np.pad(mask, ((0, ph), (0, pw)), constant_values=IGNORE_INDEX)
        h, w = mask.shape
    y, x = int(rng.integers(0, h - crop + 1)), int(rng.integers(0, w - crop + 1))
    img, mask = img[y : y + crop, x : x + crop], mask[y : y + crop, x : x + crop]
    if rng.random() < 0.5:  # horizontal flip
        img, mask = img[:, ::-1], mask[:, ::-1]
    if rng.random() < 0.5:  # vertical flip
        img, mask = img[::-1], mask[::-1]
    k = int(rng.integers(0, 4))  # rotation by k * 90 degrees (lossless for label maps)
    if k:
        img, mask = np.rot90(img, k), np.rot90(mask, k)
    return np.ascontiguousarray(img), np.ascontiguousarray(mask)


class SegDataset(Dataset):
    """Training mode: index = epoch * len + i, so augmentation is a pure function of (seed, epoch, i)
    regardless of worker count or resume point (see `EpochSampler`). Eval mode: full images, plain indices."""

    def __init__(self, samples, spec, train, crop_size=512, crops_per_image=1, seed=0):
        self.samples, self.spec, self.train = list(samples), spec, train
        self.crop_size, self.crops_per_image, self.seed = crop_size, crops_per_image, seed

    def __len__(self) -> int:
        return len(self.samples) * (self.crops_per_image if self.train else 1)

    def __getitem__(self, index: int) -> dict:
        n = len(self)
        epoch, i = divmod(index, n) if self.train else (0, index)
        s = self.samples[i % len(self.samples)]
        img = np.asarray(Image.open(s.image).convert("RGB"))
        if s.mask is not None:
            mask = decode_mask(np.asarray(Image.open(s.mask)), self.spec)
        else:
            mask = np.full(img.shape[:2], IGNORE_INDEX, dtype=np.uint8)
        if self.train:
            img, mask = _augment(img, mask, self.crop_size, np.random.default_rng([self.seed, epoch, i]))
        x = (img.astype(np.float32) / 255.0 - IMAGENET_MEAN) / IMAGENET_STD
        return {
            "image": torch.from_numpy(x).permute(2, 0, 1).contiguous(),
            "mask": torch.from_numpy(mask.astype(np.int64)),
            "domain": self.spec.domains.index(s.domain),
            "index": i % len(self.samples),
        }


class EpochSampler(Sampler):
    """One sampler spanning epochs [start, end): lets a single persistent DataLoader serve the whole run and
    makes resuming exact. Each epoch is a seeded permutation truncated to a whole number of batches."""

    def __init__(self, n: int, batch_size: int, seed: int, start_epoch: int, end_epoch: int):
        self.n, self.bs, self.seed, self.start, self.end = n, batch_size, seed, start_epoch, end_epoch

    def __len__(self) -> int:
        return (self.end - self.start) * (self.n // self.bs * self.bs)

    def __iter__(self):
        for e in range(self.start, self.end):
            perm = np.random.default_rng([self.seed, e]).permutation(self.n)[: self.n // self.bs * self.bs]
            yield from (e * self.n + int(i) for i in perm)


### LoveDA spec  
Folder layout, label mapping (raw 0 → ignore 255, raw 1–7 → classes 0–6) and a synthetic dataset generator.

In [ ]:
"""LoveDA spec: layout scanning, label mapping, and a synthetic generator for smoke tests.

Official layout (after extracting Train.zip / Val.zip / Test.zip into `data/LoveDA/`):

    <root>/{Train,Val,Test}/{Urban,Rural}/images_png/*.png
    <root>/{Train,Val}/{Urban,Rural}/masks_png/*.png      # Test masks are not public

The zips unpack to `Train/Train/...`; both nestings are accepted.

Raw mask values: 0 = no-data, 1 = background, 2 = building, 3 = road, 4 = water, 5 = barren, 6 = forest,
7 = agriculture. Training ids are raw - 1 (background is class 0, so 7 classes) and raw 0 (plus any other
unexpected value) becomes the ignore index 255.
"""

from pathlib import Path

import numpy as np
from PIL import Image


CLASS_NAMES = ("background", "building", "road", "water", "barren", "forest", "agriculture")
DOMAINS = ("urban", "rural")  # lower-case ids; directories are Urban / Rural
SPLITS = ("Train", "Val", "Test")

_decode = np.full(256, IGNORE_INDEX, dtype=np.uint8)
_decode[1:8] = np.arange(7)
_encode = np.arange(1, 8, dtype=np.uint8)


def scan_loveda(root: str | Path, split: str, require_labels: bool = True) -> list[Sample]:
    base = Path(root) / split
    if (base / split).is_dir():
        base = base / split
    samples: list[Sample] = []
    for dom in DOMAINS:
        img_dir, mask_dir = base / dom.capitalize() / "images_png", base / dom.capitalize() / "masks_png"
        if not img_dir.is_dir():
            raise FileNotFoundError(
                f"{img_dir} not found. Expected <root>/{split}/{dom.capitalize()}/images_png; "
                "see README 'Dataset' or run `python -m dl_research.prepare_dataset`."
            )
        for p in sorted(img_dir.glob("*.png")):
            m = mask_dir / p.name
            if not m.exists():
                if require_labels:
                    raise FileNotFoundError(f"missing mask for {p} (expected {m})")
                m = None
            samples.append(Sample(str(p), str(m) if m else None, dom, f"{dom}_{p.stem}"))
    if not samples:
        raise FileNotFoundError(f"no .png images found under {base}")
    return samples


def make_synthetic(root: str | Path, counts: dict[str, int] | None = None, size: int = 128, seed: int = 0) -> None:
    """Write a tiny LoveDA-shaped dataset (random images, blocky masks incl. raw 0 pixels). Test has no masks."""
    counts = counts or {"Train": 4, "Val": 4, "Test": 2}
    rng = np.random.default_rng(seed)
    for split, n in counts.items():
        for dom in DOMAINS:
            d = Path(root) / split / dom.capitalize()
            (d / "images_png").mkdir(parents=True, exist_ok=True)
            if split != "Test":
                (d / "masks_png").mkdir(parents=True, exist_ok=True)
            for i in range(n):
                blocks = rng.integers(0, 8, size=(4, 4), dtype=np.uint8)  # raw values incl. 0 (no-data)
                mask = np.kron(blocks, np.ones((size // 4, size // 4), dtype=np.uint8))
                img = (rng.random((size, size, 3)) * 80 + mask[..., None] * 20).clip(0, 255).astype(np.uint8)
                Image.fromarray(img).save(d / "images_png" / f"{i}.png")
                if split != "Test":
                    Image.fromarray(mask).save(d / "masks_png" / f"{i}.png")


LOVEDA = register(
    DatasetSpec(
        name="loveda",
        class_names=CLASS_NAMES,
        domains=DOMAINS,
        decode_lut=_decode,
        encode_lut=_encode,
        scan=scan_loveda,
        make_synthetic=make_synthetic,
    )
)


### Metrics  
Confusion-matrix based mIoU, per-class IoU, F1, pixel accuracy (ignored pixels excluded).

In [ ]:
"""Confusion-matrix metrics. Pixels whose target is the ignore index never enter the matrix."""

import numpy as np
import torch


class ConfusionMatrix:
    def __init__(self, num_classes: int, ignore_index: int = 255):
        self.n, self.ignore = num_classes, ignore_index
        self.mat = np.zeros((num_classes, num_classes), dtype=np.int64)  # rows: target, cols: prediction

    def update(self, pred: torch.Tensor, target: torch.Tensor) -> None:
        pred, target = pred.reshape(-1).long(), target.reshape(-1).long()
        valid = (target != self.ignore) & (target >= 0) & (target < self.n)
        idx = target[valid] * self.n + pred[valid].clamp(0, self.n - 1)
        self.mat += torch.bincount(idx, minlength=self.n * self.n).reshape(self.n, self.n).cpu().numpy()

    def merge(self, other: "ConfusionMatrix") -> None:
        self.mat += other.mat

    def compute(self) -> dict:
        tp = np.diag(self.mat).astype(np.float64)
        fp, fn = self.mat.sum(0) - tp, self.mat.sum(1) - tp
        with np.errstate(divide="ignore", invalid="ignore"):
            iou = np.where(tp + fp + fn > 0, tp / (tp + fp + fn), np.nan)  # NaN: class absent from GT and preds
            f1 = np.where(2 * tp + fp + fn > 0, 2 * tp / (2 * tp + fp + fn), np.nan)
        total = self.mat.sum()
        return {
            "iou": iou,
            "f1": f1,
            "miou": float(np.nanmean(iou)) if not np.all(np.isnan(iou)) else float("nan"),
            "mf1": float(np.nanmean(f1)) if not np.all(np.isnan(f1)) else float("nan"),
            "pixel_acc": float(tp.sum() / total) if total else float("nan"),
            "n_pixels": int(total),
        }

    def summary(self, class_names: tuple[str, ...]) -> dict:
        """JSON-friendly version of `compute` (NaN -> None)."""
        r = self.compute()
        nn = lambda v: None if np.isnan(v) else float(v)  # noqa: E731
        return {
            "miou": nn(r["miou"]),
            "mf1": nn(r["mf1"]),
            "pixel_acc": nn(r["pixel_acc"]),
            "n_pixels": r["n_pixels"],
            "iou": {c: nn(v) for c, v in zip(class_names, r["iou"])},
            "f1": {c: nn(v) for c, v in zip(class_names, r["f1"])},
        }


### Models  
U-Net (ResNet-50, `segmentation_models_pytorch`) and SegFormer-B2 (MiT-B2, Hugging Face).

In [ ]:
"""Model factory. Both models map (B,3,H,W) -> (B,C,H,W) logits at input resolution."""

import torch
import torch.nn as nn
import torch.nn.functional as F


class SegFormerWrapper(nn.Module):
    """HF SegFormer emits logits at 1/4 resolution; upsample bilinearly so the loss and metrics see full size."""

    def __init__(self, net: nn.Module):
        super().__init__()
        self.net = net

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        logits = self.net(pixel_values=x).logits
        return F.interpolate(logits, size=x.shape[-2:], mode="bilinear", align_corners=False)


def build_model(cfg: Config, num_classes: int) -> nn.Module:
    if cfg.model == "unet_r50":
        import segmentation_models_pytorch as smp

        return smp.Unet(
            encoder_name=cfg.encoder,
            encoder_weights="imagenet" if cfg.pretrained else None,
            in_channels=3,
            classes=num_classes,
        )
    if cfg.model == "segformer_b2":
        from transformers import SegformerConfig, SegformerForSemanticSegmentation

        if cfg.smoke:  # tiny random SegFormer: no download
            hf_cfg = SegformerConfig(
                num_labels=num_classes, depths=[1, 1, 1, 1], hidden_sizes=[16, 32, 64, 128],
                num_attention_heads=[1, 2, 4, 8], decoder_hidden_size=32,
            )
            return SegFormerWrapper(SegformerForSemanticSegmentation(hf_cfg))
        if cfg.pretrained:
            # mit-b2 checkpoint = ImageNet-1k pretrained encoder only; the all-MLP decode head is freshly initialised.
            net = SegformerForSemanticSegmentation.from_pretrained(cfg.segformer_checkpoint, num_labels=num_classes)
        else:
            hf_cfg = SegformerConfig.from_pretrained(cfg.segformer_checkpoint, num_labels=num_classes)
            net = SegformerForSemanticSegmentation(hf_cfg)
        return SegFormerWrapper(net)
    raise ValueError(f"unknown model {cfg.model!r}")


def count_params(model: nn.Module) -> dict:
    return {
        "params_total": sum(p.numel() for p in model.parameters()),
        "params_trainable": sum(p.numel() for p in model.parameters() if p.requires_grad),
    }


### Efficiency measurement  
Parameters, GFLOPs and inference speed, measured identically for both models.

In [ ]:
"""Model cost, measured the same way for every model: fp32, eval mode, no_grad, random input."""

import time

import torch
from torch.utils.flop_counter import FlopCounterMode


@torch.no_grad()
def measure_efficiency(model, device, size: int, batch_size: int = 1, warmup: int = 10, iters: int = 50) -> dict:
    model = model.to(device).eval()
    # FLOPs: torch's FlopCounterMode counts conv/matmul/attention ops (1 MAC = 2 FLOPs) for one 1 x 3 x size x size image.
    with FlopCounterMode(display=False) as fc:
        model(torch.randn(1, 3, size, size, device=device))
    x = torch.randn(batch_size, 3, size, size, device=device)
    for _ in range(warmup):
        model(x)
    sync(device)
    t0 = time.perf_counter()
    for _ in range(iters):
        model(x)
    sync(device)
    dt = (time.perf_counter() - t0) / iters
    return {
        "gflops_per_image": fc.get_total_flops() / 1e9,
        "inference_ms_per_batch": dt * 1e3,
        "inference_ms_per_image": dt * 1e3 / batch_size,
        "inference_images_per_s": batch_size / dt,
        "bench_input_size": size,
        "bench_batch_size": batch_size,
        "bench_warmup": warmup,
        "bench_iters": iters,
        "bench_precision": "fp32",
    }


### Evaluation  
Full-resolution evaluation, overall and per domain; `evaluate_run` re-evaluates a finished run.

In [ ]:
"""Evaluation on full-resolution images, overall and per domain. Also a CLI to re-evaluate a finished run.

    python -m dl_research.evaluate --run experiments/unet_r50_f1_s0 --split val [--ckpt best] [--max-save 50]

`--split test` only writes predictions: LoveDA test labels are not public, so no metrics are computed.
"""

import argparse
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from torch.utils.data import DataLoader


def prepare_data_root(cfg: Config, spec) -> str:
    """Real data root, or (smoke mode) a tiny synthetic dataset generated inside the run directory."""
    if not cfg.smoke:
        return cfg.data_root
    root = cfg.run_dir / "_synthetic_data"
    if not root.exists():
        spec.make_synthetic(root, size=2 * cfg.crop_size)
    return str(root)


def make_eval_loader(samples, spec, cfg: Config) -> DataLoader:
    ds = SegDataset(samples, spec, train=False)
    return DataLoader(ds, batch_size=cfg.eval_batch_size, shuffle=False, num_workers=cfg.num_workers)


@torch.no_grad()
def evaluate_split(model, loader, spec, device, ignore_index=255, labeled=True, save_dir=None, max_save=0, amp=False):
    """Returns {"overall": CM, "<domain>": CM, ...}. Predictions are written with RAW label values (1..7)."""
    model.eval()
    cms = {k: ConfusionMatrix(spec.num_classes, ignore_index) for k in ("overall", *spec.domains)}
    samples, saved = loader.dataset.samples, 0
    if save_dir:
        Path(save_dir).mkdir(parents=True, exist_ok=True)
    for batch in loader:
        with torch.autocast(device.type, dtype=torch.float16, enabled=amp and device.type == "cuda"):
            logits = model(batch["image"].to(device))
        pred = logits.argmax(1).cpu()
        for p, y, d, i in zip(pred, batch["mask"], batch["domain"].tolist(), batch["index"].tolist()):
            if labeled:
                cms["overall"].update(p, y)
                cms[spec.domains[d]].update(p, y)
            if save_dir and saved < max_save:
                Image.fromarray(spec.encode_lut[p.numpy()]).save(Path(save_dir) / f"{samples[i].id}.png")
                saved += 1
    return cms


def summarize(cms: dict, spec) -> dict:
    return {k: cm.summary(spec.class_names) for k, cm in cms.items()}


def evaluate_run(run, split="val", ckpt="best", max_save=None) -> dict | None:
    """Re-evaluate a finished run directory. `split="test"` writes predictions only (no labels -> no metrics)."""
    run = Path(run)
    cfg = read_config(run / "config.yaml")
    spec = get_spec(cfg.dataset)
    device = pick_device(cfg.device)
    model = build_model(cfg, spec.num_classes).to(device)
    state = torch.load(run / "checkpoints" / f"{ckpt}.pt", map_location=device)
    model.load_state_dict(state["model"])

    labeled = split == "val"
    samples = spec.scan(prepare_data_root(cfg, spec), "Val" if labeled else "Test", labeled)
    loader = make_eval_loader(samples, spec, cfg)
    max_save = len(samples) if max_save is None else max_save
    cms = evaluate_split(model, loader, spec, device, cfg.ignore_index, labeled,
                         run / "predictions" / f"{split}_{ckpt}", max_save, amp=cfg.eval_amp)
    if not labeled:
        print(f"wrote {min(max_save, len(samples))} test predictions (no labels -> no metrics)")
        return None
    out = {"split": split, "ckpt": ckpt, "epoch": state.get("epoch"), **summarize(cms, spec)}
    write_json(run / f"eval_{split}_{ckpt}.json", out)
    print(f"{split} mIoU: {out['overall']['miou']:.4f}  pixel acc: {out['overall']['pixel_acc']:.4f}")
    return out


### Training  
One (model, fraction, seed) run: AdamW + cross-entropy, resumable, writes `metrics.json`.

In [ ]:
"""Train one (model, fraction, seed) run and write metrics.json.

    python -m dl_research.train --config configs/base.yaml configs/unet_r50.yaml --set fraction=0.25 seed=1

Resumable: re-run the same command and training continues from checkpoints/last.pt (saved at every
validation epoch). A finished run (metrics.json exists) is skipped unless --force.
"""

import argparse
import json
import math
import shutil
import time
from dataclasses import replace

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader


def resolve_lr(cfg: Config) -> float:
    if cfg.lr is not None:
        return cfg.lr
    try:
        return float(read_json(cfg.tuned_lr_file)[cfg.model]["lr"])
    except (FileNotFoundError, KeyError):
        raise SystemExit(
            f"No learning rate for {cfg.model}: set lr=... or run `python -m dl_research.tune_lr` "
            f"to create {cfg.tuned_lr_file}."
        )


def masked_ce(logits, target, ignore_index):
    """Mean CE over valid pixels; a batch with no valid pixel yields 0 instead of NaN."""
    loss = F.cross_entropy(logits.float(), target, ignore_index=ignore_index, reduction="sum")
    return loss / (target != ignore_index).sum().clamp(min=1)


def lr_lambda(total_steps: int, warmup_frac: float):
    warm = max(1, int(warmup_frac * total_steps))

    def f(step: int) -> float:
        if step < warm:
            return (step + 1) / warm
        return 0.5 * (1 + math.cos(math.pi * (step - warm) / max(1, total_steps - warm)))

    return f


def eval_epochs(total: int, num_evals: int) -> set[int]:
    """1-indexed epochs after which validation runs (always includes the last)."""
    return {max(1, round(total * k / num_evals)) for k in range(1, num_evals + 1)} | {total}


def run(cfg: Config, force: bool = False) -> dict:
    run_dir = cfg.run_dir
    if (run_dir / "metrics.json").exists() and not force:
        print(f"[skip] {cfg.run_name}: already complete")
        return read_json(run_dir / "metrics.json")
    if force and run_dir.exists():
        shutil.rmtree(run_dir)
    cfg = replace(cfg, lr=resolve_lr(cfg))
    run_dir.mkdir(parents=True, exist_ok=True)
    if (run_dir / "config.yaml").exists():
        if read_config(run_dir / "config.yaml") != cfg:
            raise SystemExit(f"{run_dir}/config.yaml differs from the requested config; use --force to restart")
    else:
        dump_config(cfg, run_dir / "config.yaml")

    seed_everything(cfg.seed)
    device = pick_device(cfg.device)
    spec = get_spec(cfg.dataset)
    root = prepare_data_root(cfg, spec)
    train_samples = select_fraction(spec.scan(root, "Train", True), cfg.fraction, cfg.seed)
    val_samples = spec.scan(root, "Val", True)
    write_json(run_dir / "train_subset.json", [s.id for s in train_samples])

    train_ds = SegDataset(train_samples, spec, True, cfg.crop_size, cfg.crops_per_image, cfg.seed)
    steps_per_epoch = len(train_ds) // cfg.batch_size
    if steps_per_epoch < 1:
        raise SystemExit(f"{len(train_ds)} training crops < batch_size {cfg.batch_size}")
    E = cfg.total_epochs
    total_steps = E * steps_per_epoch

    model = build_model(cfg, spec.num_classes).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda(total_steps, cfg.warmup_frac))
    use_amp = cfg.amp and device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    start, best_miou, best_epoch, train_time, history = 0, -1.0, -1, 0.0, []
    last_path, best_path = run_dir / "checkpoints" / "last.pt", run_dir / "checkpoints" / "best.pt"
    if last_path.exists():
        ck = torch.load(last_path, map_location=device)
        model.load_state_dict(ck["model"])
        opt.load_state_dict(ck["opt"])
        sched.load_state_dict(ck["sched"])
        scaler.load_state_dict(ck["scaler"])
        start, best_miou, best_epoch = ck["epoch"], ck["best_miou"], ck["best_epoch"]
        train_time, history = ck["train_time"], ck["history"]
        print(f"[resume] {cfg.run_name} from epoch {start}/{E}")
    (run_dir / "checkpoints").mkdir(exist_ok=True)

    loader = DataLoader(
        train_ds, batch_size=cfg.batch_size, num_workers=cfg.num_workers, drop_last=True,
        sampler=EpochSampler(len(train_ds), cfg.batch_size, cfg.seed, start, E),
        persistent_workers=cfg.num_workers > 0, pin_memory=device.type == "cuda",
    )
    val_loader = make_eval_loader(val_samples, spec, cfg)
    evals = eval_epochs(E, cfg.num_evals)
    it = iter(loader)
    wall0 = time.perf_counter()

    for epoch in range(start, E):
        model.train()
        t0, losses = time.perf_counter(), []
        for _ in range(steps_per_epoch):
            b = next(it)
            x, y = b["image"].to(device, non_blocking=True), b["mask"].to(device, non_blocking=True)
            with torch.autocast(device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(x)
            loss = masked_ce(logits, y, cfg.ignore_index)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            losses.append(loss.detach())
        sync(device)
        dt = time.perf_counter() - t0  # training time excludes validation; includes data loading
        train_time += dt
        rec = {"epoch": epoch + 1, "train_loss": torch.stack(losses).mean().item(),
               "lr": sched.get_last_lr()[0], "epoch_time_s": dt}
        if epoch + 1 in evals:
            cms = evaluate_split(model, val_loader, spec, device, cfg.ignore_index, amp=cfg.eval_amp)
            rec["val_miou"] = cms["overall"].compute()["miou"]
            if rec["val_miou"] > best_miou:
                best_miou, best_epoch = rec["val_miou"], epoch + 1
                torch.save({"model": model.state_dict(), "epoch": best_epoch, "miou": best_miou}, best_path)
        history.append(rec)
        print(f"[{cfg.run_name}] " + " ".join(f"{k}={v:.4g}" if isinstance(v, float) else f"{k}={v}" for k, v in rec.items()))
        if epoch + 1 in evals:
            torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                        "scaler": scaler.state_dict(), "epoch": epoch + 1, "best_miou": best_miou,
                        "best_epoch": best_epoch, "train_time": train_time, "history": history}, last_path)
    (run_dir / "log.jsonl").write_text("".join(json.dumps(h) + "\n" for h in history))

    # Final evaluation of the best (by validation mIoU) checkpoint, with domain breakdown + sample predictions.
    model.load_state_dict(torch.load(best_path, map_location=device)["model"])
    cms = evaluate_split(model, val_loader, spec, device, cfg.ignore_index,
                         save_dir=run_dir / "predictions" / "val_best", max_save=cfg.predictions_max_images,
                         amp=cfg.eval_amp)
    bench_size = cfg.crop_size
    metrics = {
        "run": cfg.run_name,
        "complete": True,
        "config": json.loads(json.dumps(cfg.__dict__)),
        "env": env_info(device),
        "model_stats": count_params(model),
        "efficiency": measure_efficiency(model, device, bench_size, cfg.bench_batch_size,
                                         cfg.bench_warmup, cfg.bench_iters),
        "training": {
            "epochs": E, "steps_per_epoch": steps_per_epoch, "total_steps": total_steps,
            "n_train_images": len(train_samples), "n_val_images": len(val_samples),
            "train_time_s": train_time, "wall_time_this_session_s": time.perf_counter() - wall0,
            "best_epoch": best_epoch, "final_epoch_val_miou": history[-1].get("val_miou"),
            "train_batch_size": cfg.batch_size, "train_input_size": cfg.crop_size,
        },
        "val": summarize(cms, spec),
    }
    write_json(run_dir / "metrics.json", metrics)
    if cfg.prune_last_checkpoint:  # best.pt is kept; the run is complete so last.pt is no longer needed for resuming
        last_path.unlink(missing_ok=True)
    print(f"[done] {cfg.run_name}: val mIoU {metrics['val']['overall']['miou']:.4f} (best epoch {best_epoch})")
    return metrics


### Learning-rate tuning  
Grid search on the validation split only.

In [ ]:
"""Learning-rate tuning on the VALIDATION split only.

For each model, train one short run (tune_epochs, tune_fraction, tune_seed) per candidate in `lr_grids`, and keep
the LR whose best-checkpoint validation mIoU is highest. The choice is written to configs/tuned_lr.json and read
by `train` when `lr: null`.

    python -m dl_research.tune_lr --config configs/base.yaml [--models unet_r50 segformer_b2] [--set key=value ...]
"""

import argparse
from dataclasses import replace
from pathlib import Path


def tune(base: Config, models=("unet_r50", "segformer_b2")) -> dict:
    out_file = Path(base.tuned_lr_file)
    tuned = read_json(out_file) if out_file.exists() else {}
    for model in models:
        scores = {}
        for lr in base.lr_grids[model]:
            cfg = replace(
                base, model=model, lr=float(lr), fraction=base.tune_fraction, epochs=base.tune_epochs,
                scale_epochs_by_fraction=False, seed=base.tune_seed,
                out_dir=str(Path(base.out_dir) / "tune" / f"lr{lr:g}"),
            ).validate()
            scores[lr] = run(cfg)["val"]["overall"]["miou"]
        best = max(scores, key=scores.get)
        tuned[model] = {
            "lr": best,
            "val_miou": scores[best],
            "grid_val_miou": {f"{k:g}": v for k, v in scores.items()},
            "method": "grid search; one run per LR; best-checkpoint validation mIoU; validation split only",
            "tune_epochs": base.tune_epochs, "tune_fraction": base.tune_fraction, "tune_seed": base.tune_seed,
        }
        write_json(out_file, tuned)
        print(f"[tune] {model}: lr={best:g} (val mIoU {scores[best]:.4f}) -> {out_file}")
    return tuned


### Aggregation  
Mean ± std over seeds, paired model gap, tables and figures.

In [ ]:
"""Aggregate finished runs into tables (CSV + Markdown) and figures. Pure bookkeeping: it reports measured
numbers only and draws no conclusions.

    python -m dl_research.aggregate --exp-dir experiments --out results
"""

import argparse
from pathlib import Path

import matplotlib

import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402


LABELS = {"unet_r50": "U-Net (ResNet-50)", "segformer_b2": "SegFormer-B2"}
COLORS = {"unet_r50": "tab:blue", "segformer_b2": "tab:orange"}


def load_runs(exp_dir: Path) -> pd.DataFrame:
    rows = []
    for p in sorted(exp_dir.glob("*/metrics.json")):  # tuning runs live one level deeper and are excluded
        m = read_json(p)
        if not m.get("complete"):
            continue
        v, eff, tr = m["val"], m["efficiency"], m["training"]
        row = {
            "model": m["config"]["model"], "fraction": m["config"]["fraction"], "seed": m["config"]["seed"],
            "lr": m["config"]["lr"], "miou": v["overall"]["miou"], "pixel_acc": v["overall"]["pixel_acc"],
            "mf1": v["overall"]["mf1"], "urban_miou": v["urban"]["miou"], "rural_miou": v["rural"]["miou"],
            "final_epoch_miou": tr["final_epoch_val_miou"], "best_epoch": tr["best_epoch"],
            "params_m": m["model_stats"]["params_total"] / 1e6, "gflops": eff["gflops_per_image"],
            "train_time_s": tr["train_time_s"], "ms_per_image": eff["inference_ms_per_image"],
            "images_per_s": eff["inference_images_per_s"], "device": m["env"]["device_name"],
            "n_train_images": tr["n_train_images"],
        }
        row.update({f"iou_{c}": v["overall"]["iou"][c] for c in CLASS_NAMES})
        row.update({f"f1_{c}": v["overall"]["f1"][c] for c in CLASS_NAMES})
        rows.append(row)
    return pd.DataFrame(rows)


def mean_std(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    g = df.groupby(["model", "fraction"])
    out = g[cols].agg(["mean", lambda s: s.std(ddof=1)])
    out.columns = [f"{c}_{'mean' if s == 'mean' else 'std'}" for c, s in out.columns]
    out["n_seeds"] = g.size()
    return out.reset_index()


def pm(mean, std, n, digits=4) -> str:
    if pd.isna(mean):
        return "n/a"
    return f"{mean:.{digits}f} ± {std:.{digits}f}" if n > 1 and not pd.isna(std) else f"{mean:.{digits}f} (n=1)"


def md_table(df: pd.DataFrame) -> str:
    head = "| " + " | ".join(map(str, df.columns)) + " |\n|" + "---|" * len(df.columns) + "\n"
    return head + "".join("| " + " | ".join(map(str, r)) + " |\n" for r in df.itertuples(index=False))


def paired_gap(df: pd.DataFrame, col: str) -> pd.DataFrame:
    """SegFormer-B2 minus U-Net per fraction, paired by seed (only seeds present for both models)."""
    rows = []
    for frac, d in df.groupby("fraction"):
        w = d.pivot(index="seed", columns="model", values=col).dropna()
        if {"unet_r50", "segformer_b2"} <= set(w.columns) and len(w):
            diff = w["segformer_b2"] - w["unet_r50"]
            rows.append({"fraction": frac, "n_pairs": len(diff), "gap_mean": diff.mean(),
                         "gap_std": diff.std(ddof=1) if len(diff) > 1 else np.nan})
    return pd.DataFrame(rows)


def make_tables(df: pd.DataFrame, out: Path) -> str:
    out.mkdir(parents=True, exist_ok=True)
    main_cols = ["miou", "pixel_acc", "mf1", "urban_miou", "rural_miou"]
    agg = mean_std(df, main_cols + ["final_epoch_miou"] + [f"iou_{c}" for c in CLASS_NAMES])
    agg.to_csv(out / "summary.csv", index=False)
    df.to_csv(out / "all_runs.csv", index=False)

    def table(cols):
        rows = []
        for r in agg.sort_values(["fraction", "model"], ascending=[False, True]).itertuples():
            rows.append({"fraction": f"{r.fraction:g}", "model": LABELS.get(r.model, r.model), "seeds": r.n_seeds,
                         **{c: pm(getattr(r, f"{c}_mean"), getattr(r, f"{c}_std"), r.n_seeds) for c in cols}})
        return pd.DataFrame(rows)

    gap = paired_gap(df, "miou")
    gap_dom = {d: paired_gap(df, f"{d}_miou") for d in ("urban", "rural")}
    eff = df[df.fraction == df.fraction.max()].groupby("model")[["params_m", "gflops", "train_time_s", "ms_per_image",
                                                                  "images_per_s"]].mean().reset_index()
    eff["model"] = eff["model"].map(lambda m: LABELS.get(m, m))
    eff = eff.round(3)

    def gap_md(g):
        if g.empty:
            return "_no seeds with both models yet_\n"
        g = g.assign(gap=[pm(m, s, n) for m, s, n in zip(g.gap_mean, g.gap_std, g.n_pairs)])
        return md_table(g[["fraction", "n_pairs", "gap"]])

    md = ["# Results (validation split)\n",
          f"Generated from {len(df)} completed runs. mean ± sample std over seeds (ddof=1). All numbers are measured on "
          "the LoveDA **validation** split, which was also used for LR tuning and checkpoint selection (see README).\n",
          "## Overall metrics\n", md_table(table(["miou", "pixel_acc", "mf1"])),
          "\n## Urban vs rural mIoU\n", md_table(table(["urban_miou", "rural_miou"])),
          "\n## Per-class IoU\n", md_table(table([f"iou_{c}" for c in CLASS_NAMES])),
          "\n## Paired gap, SegFormer-B2 minus U-Net (mIoU, paired by seed)\n", gap_md(gap)]
    for d, g in gap_dom.items():
        md += [f"\n### {d} domain\n", gap_md(g)]
    md += [f"\n## Efficiency (runs at fraction {df.fraction.max():g}, mean over seeds)\n", md_table(eff),
           "\nGFLOPs counted by torch FlopCounterMode for one 3x{0}x{0} image (1 MAC = 2 FLOPs); inference timing is "
           "fp32, batch size 1, same device for both models. Devices: ".format(int(512)) + ", ".join(sorted(df.device.unique())) + "\n"]
    text = "".join(md)
    (out / "summary.md").write_text(text)
    gap.to_csv(out / "gap_vs_fraction.csv", index=False)
    return text


def make_plots(df: pd.DataFrame, out: Path) -> None:
    fig_dir = out / "figures"
    fig_dir.mkdir(parents=True, exist_ok=True)
    agg = mean_std(df, ["miou", "urban_miou", "rural_miou"])
    fracs = sorted(df.fraction.unique())

    fig, ax = plt.subplots(figsize=(5, 4))
    for m, d in agg.groupby("model"):
        d = d.sort_values("fraction")
        ax.errorbar(d.fraction * 100, d.miou_mean, yerr=d.miou_std.fillna(0), marker="o", capsize=3,
                    label=LABELS.get(m, m), color=COLORS.get(m))
        for _, r in df[df.model == m].iterrows():
            ax.scatter(r.fraction * 100, r.miou, s=8, color=COLORS.get(m), alpha=0.3)
    ax.set_xscale("log"); ax.set_xticks([f * 100 for f in fracs]); ax.set_xticklabels([f"{f*100:g}%" for f in fracs])
    ax.set_xlabel("training labels"); ax.set_ylabel("val mIoU"); ax.legend(); ax.grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(fig_dir / "miou_vs_fraction.png", dpi=150); plt.close(fig)

    gap = paired_gap(df, "miou")
    if not gap.empty:
        fig, ax = plt.subplots(figsize=(5, 4))
        ax.errorbar(gap.fraction * 100, gap.gap_mean, yerr=gap.gap_std.fillna(0), marker="o", capsize=3, color="k")
        ax.axhline(0, color="gray", lw=0.8); ax.set_xscale("log")
        ax.set_xticks([f * 100 for f in gap.fraction]); ax.set_xticklabels([f"{f*100:g}%" for f in gap.fraction])
        ax.set_xlabel("training labels"); ax.set_ylabel("mIoU gap (SegFormer-B2 − U-Net)"); ax.grid(alpha=0.3)
        fig.tight_layout(); fig.savefig(fig_dir / "gap_vs_fraction.png", dpi=150); plt.close(fig)

    fig, axes = plt.subplots(1, len(fracs), figsize=(3.2 * len(fracs), 4), sharey=True, squeeze=False)
    for ax, f in zip(axes[0], fracs):
        d = agg[agg.fraction == f]
        for j, dom in enumerate(("urban", "rural")):
            for k, m in enumerate(sorted(d.model)):
                r = d[d.model == m].iloc[0]
                ax.bar(j + 0.2 * (k - 0.5), r[f"{dom}_miou_mean"], 0.2, yerr=0 if pd.isna(r[f"{dom}_miou_std"]) else r[f"{dom}_miou_std"],
                       color=COLORS.get(m), label=LABELS.get(m, m) if j == 0 else None, capsize=3)
        ax.set_xticks([0, 1]); ax.set_xticklabels(["urban", "rural"]); ax.set_title(f"{f*100:g}% labels")
    axes[0][0].set_ylabel("val mIoU"); axes[0][0].legend(fontsize=7)
    fig.tight_layout(); fig.savefig(fig_dir / "domain_miou.png", dpi=150); plt.close(fig)

    top = df[df.fraction == df.fraction.max()].groupby("model")[["miou", "params_m", "gflops", "ms_per_image"]].mean()
    fig, axes = plt.subplots(1, 3, figsize=(11, 3.6))
    for ax, (col, lab) in zip(axes, [("params_m", "parameters (M)"), ("gflops", "GFLOPs / image"),
                                     ("ms_per_image", "inference ms / image")]):
        for m, r in top.iterrows():
            ax.scatter(r[col], r.miou, s=60, color=COLORS.get(m), label=LABELS.get(m, m))
        ax.set_xlabel(lab); ax.set_ylabel("val mIoU"); ax.grid(alpha=0.3)
    axes[0].legend(fontsize=7)
    fig.tight_layout(); fig.savefig(fig_dir / "efficiency.png", dpi=150); plt.close(fig)


def aggregate_runs(exp_dir="experiments", out="results") -> pd.DataFrame:
    df = load_runs(Path(exp_dir))
    if df.empty:
        raise SystemExit(f"no completed runs (metrics.json) under {exp_dir}")
    make_tables(df, Path(out))
    make_plots(df, Path(out))
    print(f"wrote tables and figures for {len(df)} runs to {out}/")
    return df


### Dataset preparation  
Unzip and verify LoveDA, write `manifest.json`. Never downloads anything.

In [ ]:
"""Check, unpack, and (opt-in) download the LoveDA dataset, and write <out>/manifest.json.

    python -m dl_research.prepare_dataset --raw ~/Downloads/LoveDA --out data/LoveDA   # unzip Train/Val/Test.zip
    python -m dl_research.prepare_dataset --out data/LoveDA                            # verify an existing layout
    python -m dl_research.prepare_dataset --out data/LoveDA --download --splits Train Val   # fetch from Zenodo
    python -m dl_research.prepare_dataset --out data/synthetic --synthetic             # tiny fake dataset

Nothing is downloaded unless you pass --download (or call `download_loveda`).
"""

import argparse
import hashlib
import os
import shutil
import subprocess
import zipfile
from pathlib import Path

import numpy as np
from PIL import Image


SPLITS = ("Train", "Val", "Test")
# Image counts of the official release as seen in a local copy; a mismatch only warns.
EXPECTED = {"Train": {"urban": 1156, "rural": 1366}, "Val": {"urban": 677, "rural": 992},
            "Test": {"urban": 820, "rural": 976}}
# Official files, Zenodo record 5706578 (open access). Sizes and MD5s are from the record's metadata.
# The record's metadata says CC-BY-4.0 but its description says CC BY-NC-SA 4.0 (non-commercial): check before use.
ZENODO = {
    "Train": ("https://zenodo.org/api/records/5706578/files/Train.zip/content", "de2b196043ed9b4af1690b3f9a7d558f", 4021669263),
    "Val": ("https://zenodo.org/api/records/5706578/files/Val.zip/content", "84cae2577468ff0b5386758bb386d31d", 2425958254),
    "Test": ("https://zenodo.org/api/records/5706578/files/Test.zip/content", "a489be0090465e01fb067795d24e6b47", 3126023212),
}


def md5sum(path, chunk: int = 1 << 24) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        while block := f.read(chunk):
            h.update(block)
    return h.hexdigest()


def download_file(url: str, dest, md5: str) -> None:
    """Resumable download with curl, then MD5 verification. A complete, valid file is not downloaded again."""
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and md5sum(dest) == md5:
        print(f"[ok] {dest.name} already downloaded and verified")
        return
    subprocess.run(["curl", "-L", "--fail", "--retry", "5", "-C", "-", "-o", str(dest), url], check=True)
    if md5sum(dest) != md5:
        raise RuntimeError(f"MD5 mismatch for {dest}; delete it and retry")
    print(f"[ok] {dest.name} downloaded and verified")


def organize(src, out) -> dict:
    """Find every <Split>/ directory (containing Urban/ and Rural/) under `src`, wherever it is nested, and move
    it to <out>/<Split>. Handles `Train/Urban`, `Train/Train/Urban` and similar re-packagings. Existing
    destinations are left alone. `src` must differ from `out`."""
    src, out = Path(src), Path(out)
    found: dict[str, Path] = {}
    for d, dirs, _ in os.walk(src):
        if {"Urban", "Rural"} <= set(dirs) and Path(d).name in SPLITS:
            found.setdefault(Path(d).name, Path(d))
    for split, d in found.items():
        dest = out / split
        if dest.exists():
            print(f"[skip] {dest} already exists")
            continue
        out.mkdir(parents=True, exist_ok=True)
        shutil.move(str(d), str(dest))
    return found


def download_loveda(out="data/LoveDA", splits=("Train", "Val"), work=None, keep_zips=False) -> None:
    """Download the official zips from Zenodo (MD5-verified), extract, and arrange as <out>/<Split>/{Urban,Rural}/...
    Test.zip (3.1 GB) has no labels and is not needed for the study, so it is not in the default `splits`."""
    out = Path(out)
    work = Path(work) if work else out.parent / f"{out.name}_download"
    for split in splits:
        if (out / split).exists():
            print(f"[skip] {out / split} already exists")
            continue
        url, md5, _ = ZENODO[split]
        zpath = work / f"{split}.zip"
        download_file(url, zpath, md5)
        ex = work / f"extracted_{split}"
        print(f"extracting {zpath.name} ...")
        with zipfile.ZipFile(zpath) as zf:
            zf.extractall(ex)
        organize(ex, out)
        shutil.rmtree(ex, ignore_errors=True)
        if not keep_zips:
            zpath.unlink(missing_ok=True)


def prepare(out="data/LoveDA", raw=None, synthetic=False, download=False, splits=SPLITS) -> dict:
    spec, out = get_spec("loveda"), Path(out)

    if synthetic:
        spec.make_synthetic(out)
    elif download:
        download_loveda(out, splits)
    elif raw:
        for split in SPLITS:
            z = Path(raw) / f"{split}.zip"
            if z.exists():
                print(f"extracting {z} -> {out}")
                with zipfile.ZipFile(z) as zf:
                    zf.extractall(out)
            else:
                print(f"warning: {z} not found, skipping")

    manifest, problems = {}, []
    for split in splits:
        try:
            samples = spec.scan(out, split, require_labels=split != "Test")
        except FileNotFoundError as e:
            problems.append(str(e))
            continue
        counts = {d: sum(s.domain == d for s in samples) for d in spec.domains}
        manifest[split] = {"counts": counts, "labeled": all(s.mask for s in samples)}
        if not synthetic and counts != EXPECTED[split]:
            print(f"warning: {split} counts {counts} differ from expected {EXPECTED[split]}")
        if split != "Test":  # sanity-check label values on a sample of masks
            vals = set()
            for s in samples[:: max(1, len(samples) // 50)]:
                vals |= set(np.unique(np.asarray(Image.open(s.mask))).tolist())
            manifest[split]["raw_mask_values_seen"] = sorted(vals)
            if not vals <= set(range(8)):
                print(f"warning: {split} masks contain values outside 0..7: {sorted(vals - set(range(8)))}")
        print(f"{split}: {counts}")
    if problems:
        raise SystemExit("\n".join(problems))
    write_json(out / "manifest.json", manifest)
    print(f"ok -> {out / 'manifest.json'}")
    return manifest


## 3. Experiment settings
The shared protocol is the `Config` defaults; change a value here and it applies to both models.

In [ ]:
import sys
from dataclasses import replace
from pathlib import Path

cfg = build_config({
    "data_root": "data/LoveDA",            # see "Dataset" cell
    "out_dir": "experiments",
    "tuned_lr_file": "tuned_lr.json",
    # DataLoader workers need importable classes; notebook-defined classes can't be pickled for spawn (macOS/Windows).
    "num_workers": 4 if sys.platform.startswith("linux") else 0,
    "lr_grids": {"unet_r50": [1e-4, 3e-4, 1e-3], "segformer_b2": [3e-5, 6e-5, 1e-4, 3e-4]},
})
MODELS, SEEDS, FRACTIONS = ["unet_r50", "segformer_b2"], [0, 1, 2], [1.0, 0.25, 0.1]
cfg

## 4. Smoke test
Checks the whole pipeline on synthetic data. Metrics from this are meaningless — it only proves the code runs.

In [ ]:
# Tiny synthetic end-to-end check (CPU, no dataset, no downloads, about a minute).
RUN_SMOKE = True
if RUN_SMOKE:
    smoke = replace(cfg, smoke=True, pretrained=False, encoder="resnet18", crop_size=64, batch_size=2,
                    eval_batch_size=2, num_workers=0, epochs=2, num_evals=2, lr=1e-3, amp=False, device="cpu",
                    predictions_max_images=2, bench_warmup=1, bench_iters=2, out_dir="experiments/smoke").validate()
    for m in MODELS:
        run(replace(smoke, model=m))
    aggregate_runs("experiments/smoke", "experiments/smoke/results")

## 5. Dataset
Expected layout: `data/LoveDA/{Train,Val,Test}/{Urban,Rural}/images_png` (+ `masks_png` for Train/Val).

In [ ]:
# Put Train.zip / Val.zip / Test.zip (official LoveDA release) in RAW_DIR, or extract them under cfg.data_root yourself.
RUN_PREPARE = False
RAW_DIR = None   # e.g. "~/Downloads/LoveDA"
if RUN_PREPARE:
    prepare(out=cfg.data_root, raw=str(Path(RAW_DIR).expanduser()) if RAW_DIR else None)

## 6. Learning-rate tuning (validation split only)

In [ ]:
# Short validation-only LR grid search per model -> tuned_lr.json (used by every study run).
RUN_TUNING = False
if RUN_TUNING:
    tuned = tune(cfg, MODELS)

## 7. Full study
2 models × 3 label fractions × 3 seeds is a long run; set `RUN_STUDY = True` deliberately.

In [ ]:
# Full study: models x fractions x seeds. Resumable: re-run the cell and finished runs are skipped.
RUN_STUDY = False
if RUN_STUDY:
    for seed in SEEDS:
        for frac in FRACTIONS:
            for m in MODELS:
                run(replace(cfg, model=m, fraction=frac, seed=seed))

## 8. Results
Tables and figures come only from completed runs; no conclusions are written automatically.

In [ ]:
from IPython.display import Image as IPImage, Markdown, display

if any(Path("experiments").glob("*/metrics.json")):
    df = aggregate_runs("experiments", "results")
    display(Markdown(Path("results/summary.md").read_text()))
    for fig in sorted(Path("results/figures").glob("*.png")):
        display(IPImage(filename=str(fig)))
else:
    print("No completed study runs yet: enable RUN_TUNING / RUN_STUDY above first.")

## 9. Re-evaluation

In [ ]:
# Re-evaluate a finished run (best checkpoint) on val, saving all predictions; use split="test" for prediction-only output.
# evaluate_run("experiments/segformer_b2_f0.25_s1", split="val", max_save=100)